# 🧠 Day 01a: Memory Basics — Allocation & Fragmentation

---

## 🎯 What You'll Master Today
- Contiguous vs non-contiguous memory allocation
- Internal vs external fragmentation
- First-fit, best-fit, worst-fit algorithms
- Compaction and memory partitioning

---

## 🏬 The Parking Lot Analogy

Memory allocation is like a parking lot:
- **Contiguous** = you need the ENTIRE row of spots together
- **Non-contiguous** = park across multiple rows, just remember where
- **External fragmentation** = scattered empty spots that no big car can fit
- **Internal fragmentation** = SUV spot assigned to a motorcycle, space wasted

```
╔══════════════════════════════════════════════════════╗
║           MEMORY ALLOCATION STRATEGIES               ║
╠══════════════════════════════════════════════════════╣
║                                                      ║
║  Fixed Partition:  |  P1  |  P2  | FREE |  P3  |    ║
║                    equal/unequal sizes, internal frag║
║                                                      ║
║  Variable Partition:|P1|  P2  |FREE|P3| FREE |      ║
║                    exact fit, external frag          ║
║                                                      ║
║  Paging/Segmentation → non-contiguous (next topic)  ║
║                                                      ║
╚══════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Memory Allocation Strategies Simulator
# ============================================

class MemoryAllocator:
    """Simulates First-Fit, Best-Fit, Worst-Fit allocation."""
    
    def __init__(self, total_memory: int, partitions: list[int]):
        # Each partition: [size, process_name or None]
        self.partitions = [[size, None] for size in partitions]
        self.total = total_memory
    
    def first_fit(self, process: str, size: int) -> bool:
        """Allocate to FIRST partition that fits."""
        for part in self.partitions:
            if part[1] is None and part[0] >= size:
                part[1] = process
                print(f"  First-Fit: {process}({size}KB) → partition {part[0]}KB")
                return True
        print(f"  First-Fit: {process}({size}KB) → NO FIT ❌")
        return False
    
    def best_fit(self, process: str, size: int) -> bool:
        """Allocate to SMALLEST partition that fits."""
        best_idx = -1
        best_waste = float('inf')
        for i, part in enumerate(self.partitions):
            if part[1] is None and part[0] >= size:
                waste = part[0] - size
                if waste < best_waste:
                    best_waste = waste
                    best_idx = i
        if best_idx != -1:
            self.partitions[best_idx][1] = process
            print(f"  Best-Fit: {process}({size}KB) → partition {self.partitions[best_idx][0]}KB (waste: {best_waste}KB)")
            return True
        print(f"  Best-Fit: {process}({size}KB) → NO FIT ❌")
        return False
    
    def worst_fit(self, process: str, size: int) -> bool:
        """Allocate to LARGEST partition (minimize leftover unusable holes)."""
        worst_idx = -1
        worst_size = -1
        for i, part in enumerate(self.partitions):
            if part[1] is None and part[0] >= size:
                if part[0] > worst_size:
                    worst_size = part[0]
                    worst_idx = i
        if worst_idx != -1:
            self.partitions[worst_idx][1] = process
            print(f"  Worst-Fit: {process}({size}KB) → partition {self.partitions[worst_idx][0]}KB")
            return True
        print(f"  Worst-Fit: {process}({size}KB) → NO FIT ❌")
        return False
    
    def display(self):
        print("  Memory Layout:")
        for i, (size, proc) in enumerate(self.partitions):
            status = proc if proc else "FREE"
            bar = '█' * (size // 10) if proc else '░' * (size // 10)
            print(f"    [{i}] {bar} {size}KB → {status}")
    
    def reset(self):
        for part in self.partitions:
            part[1] = None


# Compare all 3 strategies
partitions = [100, 500, 200, 300, 600]
processes = [("P1", 212), ("P2", 417), ("P3", 112), ("P4", 426)]

for strategy in ['first_fit', 'best_fit', 'worst_fit']:
    print(f"\n{'='*50}")
    print(f"  Strategy: {strategy.upper()}")
    print(f"{'='*50}")
    alloc = MemoryAllocator(1700, partitions)
    for name, size in processes:
        getattr(alloc, strategy)(name, size)
    alloc.display()

In [ ]:
# ============================================
# 2. Fragmentation Demo
# ============================================

def fragmentation_analysis(partitions: list[int], process_sizes: list[int]):
    """
    Show internal & external fragmentation for fixed partitioning.
    """
    print("  Fixed Partition Fragmentation Analysis")
    print("  " + "─" * 40)
    
    internal_frag = 0
    external_frag_partitions = []
    
    for i, (part, proc) in enumerate(zip(partitions, process_sizes + [0] * len(partitions))):
        if proc > 0 and proc <= part:
            waste = part - proc
            internal_frag += waste
            print(f"    Partition {part}KB ← Process {proc}KB | Internal frag: {waste}KB")
        elif proc == 0:
            external_frag_partitions.append(part)
            print(f"    Partition {part}KB ← EMPTY")
        else:
            print(f"    Partition {part}KB ← Process {proc}KB WON'T FIT ❌")
    
    ext_total = sum(external_frag_partitions)
    print(f"\n  📊 Internal Fragmentation: {internal_frag}KB")
    print(f"  📊 External Free Space: {ext_total}KB (in {len(external_frag_partitions)} blocks)")
    print(f"  💡 A process needing {ext_total}KB could fit if memory were contiguous!")


# Fixed partitions: 200, 400, 600, 500, 300
# Processes arrive: 180, 350, 400
fragmentation_analysis(
    partitions=[200, 400, 600, 500, 300],
    process_sizes=[180, 350, 400]
)

```
╔══════════════════════════════════════════════════════════════╗
║           ALLOCATION STRATEGY COMPARISON                     ║
╠════════════╦═══════════════╦═════════════════════════════════╣
║  Strategy  ║  Speed        ║  Fragmentation                  ║
╠════════════╬═══════════════╬═════════════════════════════════╣
║  First-Fit ║  O(n) fast    ║  Moderate external frag         ║
║  Best-Fit  ║  O(n) scan    ║  Tiny leftover holes (bad!)     ║
║  Worst-Fit ║  O(n) scan    ║  Bigger leftovers (usable)      ║
║  Next-Fit  ║  O(n) resume  ║  Spreads allocation evenly      ║
╚════════════╩═══════════════╩═════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Internal vs external fragmentation? | Internal: wasted space INSIDE partition. External: free space scattered, can't serve big request |
| 2 | Which fit is best in practice? | First-fit is fastest with comparable fragmentation. Best-fit creates tiny unusable holes |
| 3 | How to solve external fragmentation? | Compaction (expensive), or paging (non-contiguous allocation) |
| 4 | Fixed vs variable partitioning? | Fixed: simple, internal frag. Variable: flexible, external frag |
| 5 | What is compaction? | Shuffle processes in memory to merge free holes. Needs relocation, expensive |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Fixed partition → internal frag, variable → external │
## │  • First-fit: fast & good enough for most cases          │
## │  • Best-fit: smallest hole → tiny unusable leftovers     │
## │  • Paging solves external fragmentation (next topic)     │
## │  • Compaction = defragmentation, O(n) and needs pausing │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Paging** — non-contiguous allocation with page tables